# 01. Probability Distribution

Agent jarang bekerja dengan informasi lengkap: kamera terhalang hujan, sensor bapuk, dll. Logika (benar/salah) tidak cukup untuk kondisi ini. **Probabilitas** memberi angka 0-1 untuk derajat keyakinan agent terhadap suatu kejadian, sehingga agent tetap bisa memilih aksi terbaik walau tidak pasti.

Setelah notebook ini, mahasiswa diharapkan mampu:
1. menghitung mean, variance, dan standard deviation, serta membedakan parameter populasi dan statistik sampel;
2. memilih distribusi diskrit/kontinu yang cocok untuk suatu kasus dan membaca PMF, PDF, dan CDF;
3. menghitung probabilitas bersyarat dan menerapkan aturan Bayes pada kasus dua variabel.

## Setup

Jalankan sel ini lebih dahulu. Notebook ini memakai `scipy.stats` untuk semua distribusi.

In [ ]:
# Standard setup cell - run before other cells.
import importlib.util
import subprocess
import sys

def ensure_dependencies():
    """Install only packages used by this module."""
    required = {"numpy": "numpy", "matplotlib": "matplotlib", "pandas": "pandas", "scipy": "scipy"}
    missing = [pkg for mod, pkg in required.items() if importlib.util.find_spec(mod) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

ensure_dependencies()
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy import stats

rng = np.random.default_rng(42)

---
## 1.1 Variabel acak dan notasi

| Notasi | Arti | Contoh (lempar dadu) |
|---|---|---|
| $Y$ (huruf besar) | **variabel acak**: besaran yang hasilnya belum diketahui | hasil lemparan |
| $y$ (huruf kecil) | satu **nilai spesifik** yang mungkin muncul | $y=3$ |
| $P(Y=y)$ atau $p(y)$ | peluang $Y$ bernilai $y$ | $P(Y=3)=1/6$ |
| $\Omega$ | ruang sampel, semua hasil yang mungkin | $\{1,\dots,6\}$ |

Syarat setiap distribusi: $0\le p(y)\le 1$ dan $\sum_y p(y)=1$ (diskrit) atau $\int f(y)\,dy=1$ (kontinu).

---
## 1.2 Parameter data: mean, variance, standard deviation

| Ukuran | Populasi (parameter) | Sampel (statistik) |
|---|---|---|
| Rata-rata | $\mu = E[Y] = \sum_y y\,p(y)$ | $\bar{x} = \frac{1}{n}\sum_i x_i$ |
| Varians | $\sigma^2 = E[(Y-\mu)^2] = E[Y^2]-\mu^2$ | $s^2 = \frac{1}{n-1}\sum_i (x_i-\bar{x})^2$ |
| Simpangan baku | $\sigma=\sqrt{\sigma^2}$ | $s=\sqrt{s^2}$ |

- **Mean** = pusat data; **variance** = rata-rata kuadrat jarak ke pusat (satuannya kuadrat); **std dev** = sebaran dalam satuan asli.
- Populasi = seluruh objek (biasanya tidak bisa diukur semua). Sampel = sebagian yang diamati, dipakai untuk **menaksir** $\mu$ dan $\sigma^2$.
- Pembagi $n-1$ (*Bessel's correction*) dipakai karena $\bar{x}$ dihitung dari sampel yang sama, sehingga sebaran cenderung terlalu kecil bila dibagi $n$. Di NumPy: `np.var(x, ddof=1)`.

![populasi vs sampel](img/population-vs-sample.png)

### Contoh penerapan

In [ ]:
# Populasi: dadu adil, dihitung langsung dari distribusinya
y = np.arange(1, 7)
p = np.full(6, 1 / 6)
mu = np.sum(y * p)
var = np.sum(y**2 * p) - mu**2          # E[Y^2] - mu^2
print(f"Populasi : mu = {mu:.4f}, sigma^2 = {var:.4f}, sigma = {np.sqrt(var):.4f}")

# Sampel: 20 lemparan
x = rng.integers(1, 7, size=20)
print(f"Sampel   : x = {x}")
print(f"           x_bar = {x.mean():.4f}, s^2 = {x.var(ddof=1):.4f}, s = {x.std(ddof=1):.4f}")

$\mu=3{,}5$ dan $\sigma^2=91/6-3{,}5^2\approx 2{,}9167$ adalah nilai pasti dari distribusi. $\bar{x}$ dan $s^2$ berubah tiap kali sampel diambil ulang dan makin dekat ke $\mu,\sigma^2$ ketika $n$ membesar.

---
## 1.3 Distribusi diskrit

Hasilnya **dapat dihitung** (*countable*). Peluang tiap nilai diberikan oleh **PMF** (*probability mass function*) $p(y)=P(Y=y)$.

| Distribusi | PMF $p(y)$ | $E[Y]$ | $\mathrm{Var}(Y)$ | Kapan dipakai / contoh |
|---|---|---|---|---|
| **Uniform** $U(a,b)$ | $\frac{1}{b-a+1}$, $y\in\{a,\dots,b\}$ | $\frac{a+b}{2}$ | $\frac{(b-a+1)^2-1}{12}$ | semua hasil sama peluang (*equiprobable*): dadu |
| **Bernoulli** $Bern(p)$ | $p^y(1-p)^{1-y}$, $y\in\{0,1\}$ | $p$ | $p(1-p)$ | **1 kali** percobaan sukses/gagal: klik iklan atau tidak |
| **Binomial** $B(n,p)$ | $\binom{n}{y}p^y(1-p)^{n-y}$ | $np$ | $np(1-p)$ | jumlah sukses dari $n$ Bernoulli **independen**: hari harga saham naik dalam 5 hari bursa |
| **Poisson** $Po(\lambda)$ | $\frac{\lambda^y e^{-\lambda}}{y!}$ | $\lambda$ | $\lambda$ | banyak kejadian per interval waktu/ruang: pertanyaan masuk ke chatbot per jam |

![distribusi diskrit](img/discrete-distributions.png)

Bernoulli adalah Binomial dengan $n=1$. Poisson mendekati Binomial bila $n$ besar dan $p$ kecil, dengan $\lambda=np$.

### Contoh penerapan

In [ ]:
# Harga saham naik pada satu hari dengan p = 0,6. Berapa peluang naik tepat 3 dari 5 hari?
n, p_up = 5, 0.6
B = stats.binom(n, p_up)
print(f"P(Y=3)  = {B.pmf(3):.4f}")
print(f"P(Y>=4) = {B.sf(3):.4f}")   # sf(k) = P(Y > k)

# Chatbot menerima rata-rata 4 pertanyaan per jam. Peluang lonjakan >= 8 pertanyaan?
Po = stats.poisson(4)
print(f"P(Y>=8) = {Po.sf(7):.4f}")

# Cek rumus E dan Var dengan simulasi 100.000 sampel
dists = {"Uniform U(1,6)": stats.randint(1, 7), "Bern(0.7)": stats.bernoulli(0.7),
         "B(5, 0.6)": B, "Po(4)": Po}
display(pd.DataFrame([
    {"distribusi": name, "E teori": d.mean(), "Var teori": d.var(),
     "mean sim": (s := d.rvs(100_000, random_state=rng)).mean(), "var sim": s.var(ddof=1)}
    for name, d in dists.items()
]).round(4))

Kolom simulasi hampir sama dengan kolom teori. Perhatikan `stats.randint(1, 7)`: batas atas SciPy bersifat eksklusif, jadi ini dadu $\{1,\dots,6\}$.

---
## 1.4 Distribusi kontinu

Nilai $Y$ berada pada rentang kontinu, sehingga ada tak hingga kemungkinan dan **$P(Y=y)=0$ untuk satu titik**. Yang bermakna adalah peluang pada **interval**:

$$P(a\le Y\le b)=\int_a^b f(y)\,dy = F(b)-F(a),\qquad F(y)=P(Y\le y)$$

$f(y)$ adalah **PDF** (*probability density function*, tinggi kurva, boleh > 1), $F(y)$ adalah **CDF** (*cumulative distribution function*, selalu naik dari 0 ke 1).

![pdf vs cdf](img/pdf-vs-cdf.png)

| Distribusi | Parameter | $E[Y]$ | $\mathrm{Var}(Y)$ | Ciri / contoh |
|---|---|---|---|---|
| **Normal** $N(\mu,\sigma^2)$ | $\mu,\sigma^2$ | $\mu$ | $\sigma^2$ | lonceng simetris; tinggi badan, derau sensor |
| **Standard Normal** $Z$ | - | $0$ | $1$ | hasil standardisasi $Z=\frac{Y-\mu}{\sigma}$ |
| **Student's t** $t(df)$ | $df=n-1$ | $0$ ($df>1$) | $\frac{df}{df-2}$ ($df>2$) | sampel kecil ($n<30$); ekor tebal mengantisipasi *outlier*; $\to Z$ bila $df\to\infty$ |
| **Chi-squared** $\chi^2(k)$ | $k$ | $k$ | $2k$ | non-negatif, miring ke kanan; uji *goodness of fit* |
| **Exponential** $Exp(\lambda)$ | $\lambda$ (laju) | $1/\lambda$ | $1/\lambda^2$ | turun cepat lalu melandai; *views* video setelah upload, waktu tunggu antar-kejadian Poisson |
| **Logistic** $Logistic(\mu,s)$ | $\mu,s$ | $\mu$ | $\frac{s^2\pi^2}{3}$ | CDF-nya kurva S $\frac{1}{1+e^{-(y-\mu)/s}}$; input kontinu → peluang biner (menang/kalah e-sports) |

![distribusi kontinu](img/continuous-distributions.png)

### Normal, aturan empiris, dan Z-score

Untuk data normal, sekitar **68%** data berada dalam $\mu\pm\sigma$, **95%** dalam $\mu\pm2\sigma$, dan **99,7%** dalam $\mu\pm3\sigma$. Z-score mengubah nilai apa pun menjadi "berapa $\sigma$ dari mean", sehingga data berbeda satuan bisa dibandingkan.

![aturan empiris](img/empirical-rule.png)

### Contoh penerapan

In [ ]:
# Aturan empiris dari CDF normal baku
for k in (1, 2, 3):
    print(f"P(|Z| <= {k}) = {stats.norm.cdf(k) - stats.norm.cdf(-k):.4f}")

# Tinggi mahasiswa ~ N(170, 8^2). Peluang tinggi antara 160 dan 180 cm?
mu_h, sigma_h = 170, 8
z_lo, z_hi = (160 - mu_h) / sigma_h, (180 - mu_h) / sigma_h
print(f"\nZ = [{z_lo}, {z_hi}] -> P = {stats.norm.cdf(z_hi) - stats.norm.cdf(z_lo):.4f}")
print(f"Langsung N(170, 8): P = {stats.norm(mu_h, sigma_h).cdf(180) - stats.norm(mu_h, sigma_h).cdf(160):.4f}")
print(f"P(Y = 170) = 0, tetapi f(170) = {stats.norm(mu_h, sigma_h).pdf(170):.4f} (kepadatan, bukan peluang)")

# Ekor tebal: peluang |nilai| > 3 pada Normal vs t
for name, d in [("N(0,1)", stats.norm), ("t(df=4)", stats.t(4))]:
    print(f"P(|Y| > 3) {name:8s}= {2 * d.sf(3):.4f}")

Z-score $[-1{,}25;\ 1{,}25]$ memberi peluang yang sama dengan menghitung langsung pada $N(170,8^2)$. Distribusi $t(4)$ memberi peluang nilai ekstrem sekitar 15 kali lebih besar daripada normal, itulah arti "ekor tebal".

---
## 1.5 Joint, conditional, dan aturan Bayes

Contoh klasik (Russell & Norvig): dua variabel Boolean, *Cavity* (gigi berlubang) dan *Toothache* (sakit gigi). Tabel **joint distribution** berisi peluang setiap kombinasi, jumlahnya 1:

| | toothache | ¬toothache | **marginal** |
|---|---|---|---|
| **cavity** | 0,12 | 0,08 | $P(cavity)=0{,}20$ |
| **¬cavity** | 0,08 | 0,72 | 0,80 |
| **marginal** | $P(toothache)=0{,}20$ | 0,80 | 1 |

Begitula

| Konsep | Rumus | Contoh |
|---|---|---|
| Joint | $P(A\cap B)$ | $P(cavity \wedge toothache)=0{,}12$ |
| Marginal | $P(A)=\sum_b P(A, b)$ | $P(cavity)=0{,}12+0{,}08=0{,}20$ |
| Conditional | $P(A\mid B)=\dfrac{P(A\cap B)}{P(B)}$ | $P(cavity\mid toothache)=0{,}12/0{,}20=0{,}6$ |
| Product rule | $P(A\cap B)=P(A\mid B)P(B)=P(B\mid A)P(A)$ | |
| Independen | $P(A\mid B)=P(A)$ | tidak berlaku di sini: $0{,}6\ne0{,}2$ |

### Aturan Bayes

Dari product rule diperoleh:

$$\underbrace{P(H\mid E)}_{\text{posterior}}=\frac{\overbrace{P(E\mid H)}^{\text{likelihood}}\;\overbrace{P(H)}^{\text{prior}}}{\underbrace{P(E)}_{\text{evidence}}},\qquad P(E)=P(E\mid H)P(H)+P(E\mid\neg H)P(\neg H)$$

Agent memulai dari keyakinan awal (**prior**), mengamati bukti (**evidence**), lalu memperbarui keyakinan (**posterior**). Bayes berguna karena $P(E\mid H)$ (arah sebab → gejala) biasanya lebih mudah diketahui daripada $P(H\mid E)$ (gejala → sebab) yang dibutuhkan untuk diagnosis.

**Contoh tes medis.** Prevalensi penyakit 1%, sensitivitas tes 90% ($P(+\mid sakit)$), *false positive rate* 9% ($P(+\mid sehat)$). Hasil tes positif; berapa peluang benar-benar sakit?

![bayes tes medis](img/bayes-medical-test.png)

Posterior hanya ~9% karena prior sangat kecil: dari 1000 orang, positif palsu (89) jauh lebih banyak daripada positif benar (9). Mengabaikan prior disebut *base rate fallacy*.

**Conditional independence.** $A$ dan $B$ independen bersyarat $C$ bila $P(A\cap B\mid C)=P(A\mid C)\,P(B\mid C)$. Contoh: setelah diketahui ada *cavity*, sakit gigi dan alat dokter yang "nyangkut" (*catch*) tidak saling memberi informasi lagi. Ini memangkas jumlah angka yang perlu disimpan agent dan menjadi dasar model probabilistik di modul berikutnya.

**Di AI:** diagnosis medis, deteksi objek mobil otonom (gabungan bukti sensor), filter spam/rekomendasi (Naive Bayes), dan **confidence threshold**: agent bertindak hanya jika posterior melewati ambang, misalnya rem darurat bila $P(pejalan\ kaki\mid sensor)>0{,}8$.

### Contoh penerapan

In [ ]:
joint = pd.DataFrame([[0.12, 0.08], [0.08, 0.72]],
                     index=["cavity", "no_cavity"], columns=["toothache", "no_toothache"])
p_toothache = joint["toothache"].sum()
p_cavity = joint.loc["cavity"].sum()
print(f"P(toothache)          = {p_toothache:.2f}")
print(f"P(cavity | toothache) = {joint.loc['cavity', 'toothache'] / p_toothache:.2f}")

def bayes(prior, likelihood, false_pos):
    """Posterior P(H|E) untuk satu hipotesis biner."""
    evidence = likelihood * prior + false_pos * (1 - prior)
    return likelihood * prior / evidence

# Cara Bayes untuk cavity: P(toothache|cavity) dan P(toothache|no_cavity) dari tabel
lik = joint.loc["cavity", "toothache"] / p_cavity
fp = joint.loc["no_cavity", "toothache"] / (1 - p_cavity)
print(f"Bayes                 = {bayes(p_cavity, lik, fp):.2f}")

# Tes medis
post = bayes(prior=0.01, likelihood=0.90, false_pos=0.09)
print(f"\nP(sakit | +) = {post:.4f}")

# Tes kedua (independen bersyarat status penyakit): posterior lama menjadi prior baru
post2 = bayes(prior=post, likelihood=0.90, false_pos=0.09)
print(f"P(sakit | +, +) = {post2:.4f}")

Kedua cara memberi $P(cavity\mid toothache)=0{,}6$. Pada tes medis, satu hasil positif menaikkan keyakinan dari 1% ke ~9%, dan tes positif kedua menaikkannya ke ~50%. Posterior satu langkah menjadi prior langkah berikutnya, ini inti *belief update* pada agent.

---
## Rangkuman

| Topik | Inti |
|---|---|
| Parameter | $\mu$, $\sigma^2=E[Y^2]-\mu^2$, $\sigma$; sampel memakai $\bar{x}$ dan $s^2$ (pembagi $n-1$) |
| Diskrit | PMF; Uniform, Bernoulli, Binomial, Poisson |
| Kontinu | PDF/CDF, $P(Y=y)=0$; Normal, $Z$, t, $\chi^2$, Exponential, Logistic |
| Bayes | posterior ∝ likelihood × prior; prior kecil membuat posterior tetap kecil |

### Soal 1 - Parameter data

Waktu respons server (ms) dari 8 permintaan: `[120, 135, 128, 410, 131, 125, 140, 122]`.

1. Hitung $\bar{x}$, $s^2$, dan $s$ secara manual, lalu verifikasi dengan NumPy.
2. Hitung juga varians dengan pembagi $n$. Mana yang tepat dipakai di sini dan mengapa?
3. Hitung Z-score setiap data (pakai $\bar{x}$ dan $s$). Data mana yang mencurigakan? Apa pengaruhnya terhadap $\bar{x}$ dan $s$ jika data itu dibuang?

<details>
<summary>Klik untuk melihat hint</summary>

Data ini adalah sampel, bukan seluruh populasi permintaan. Untuk bagian 3, bandingkan $|Z|$ dengan batas pada aturan empiris, lalu hitung ulang statistik tanpa data tersebut dan perhatikan ukuran mana yang paling berubah.

</details>

### Soal 2 - Memilih distribusi

Untuk setiap kasus, tentukan distribusi dan parameternya, lalu hitung peluang yang diminta dengan `scipy.stats`.

1. Model klasifikasi gambar benar dengan peluang 0,85 per gambar. Dari 20 gambar, berapa peluang **paling sedikit** 18 benar?
2. Server menerima rata-rata 3 *error log* per menit. Berapa peluang dalam satu menit **tidak ada** error? Berapa peluang **lebih dari 6**?
3. Nilai ujian ~ $N(72, 10^2)$. Berapa persen mahasiswa mendapat nilai di atas 90? Nilai minimal berapa agar masuk 10% teratas?
4. Waktu sampai error berikutnya pada kasus 2 mengikuti $Exp(\lambda)$. Berapa peluang menunggu lebih dari 1 menit? Bandingkan dengan jawaban "tidak ada error" pada nomor 2 dan jelaskan.

<details>
<summary>Klik untuk melihat hint</summary>

Bedakan "jumlah sukses dari $n$ percobaan" dengan "jumlah kejadian per interval". Gunakan `sf` untuk ekor kanan dan `ppf` (invers CDF) untuk mencari nilai batas persentil. Di SciPy, `stats.expon` memakai `scale` $=1/\lambda$.

</details>

### Soal 3 - Bayes dan confidence threshold

Sebuah filter spam mendeteksi kata "gratis". Diketahui 30% email adalah spam, $P(\text{gratis}\mid spam)=0{,}6$, dan $P(\text{gratis}\mid \neg spam)=0{,}05$.

1. Susun tabel joint $2\times2$ untuk *Spam* dan *Gratis*, lalu hitung $P(\text{gratis})$.
2. Hitung $P(spam\mid \text{gratis})$ dan $P(spam\mid \neg\text{gratis})$.
3. Filter memindahkan email ke folder spam hanya jika posterior > 0,9. Apakah email berisi "gratis" dipindahkan? Berapa nilai $P(\text{gratis}\mid\neg spam)$ maksimum agar email itu dipindahkan?
4. Jelaskan mengapa menaikkan threshold mengurangi *false positive* tetapi menambah spam yang lolos.

<details>
<summary>Klik untuk melihat hint</summary>

Isi tabel joint dengan product rule $P(A\cap B)=P(B\mid A)P(A)$; keempat sel harus berjumlah 1. Untuk bagian 3, tulis posterior sebagai fungsi dari $P(\text{gratis}\mid\neg spam)$, set sama dengan 0,9, lalu selesaikan.

</details>